# Deploy to AgentCore Runtime

> **Hosted Workshop Note:** During hosted OneBlink workshops, SageMaker roles do not have the IAM permissions required to run this notebook. You can review the code and concepts here, but deployment will only work in your own AWS account with the appropriate permissions configured.

This notebook deploys the Strands agent from `01_basic_strands_agent.ipynb` as a **managed service** on Amazon Bedrock AgentCore Runtime. The agent code and dependencies are pre-built in the `agentcore_deploy/` directory.

**Architecture:**
```
User → AgentCore Runtime → Strands Agent (Claude) → Tools
```

You'll learn how to:
- Configure the AgentCore deployment
- Deploy using `direct_code_deploy` (no Docker required)
- Invoke the deployed agent via CLI and boto3

**Prerequisites:** Complete `01_basic_strands_agent.ipynb` first. Ensure `../CONFIG.txt` is configured.

## 1. Setup

Install the AgentCore starter toolkit and dependencies, then verify your SageMaker role has the required deployment permissions.

In [ ]:
%pip install bedrock-agentcore-starter-toolkit>=0.3.3 bedrock-agentcore>=1.4.7 pyyaml python-dotenv -q

In [ ]:
import sys

import boto3

# Detect current SageMaker execution role
sts = boto3.client("sts")
caller = sts.get_caller_identity()
account_id = caller["Account"]
role_name = caller["Arn"].split("/")[1]

# Check if the deployment policy is attached
POLICY_NAME = "BedrockAgentCoreLabDeployPolicy"
POLICY_ARN = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"

iam = boto3.client("iam")
attached = iam.list_attached_role_policies(RoleName=role_name)
policy_attached = any(
    p["PolicyArn"] == POLICY_ARN for p in attached["AttachedPolicies"]
)

if policy_attached:
    print(f"Deployment policy is attached to {role_name}")
else:
    print(f"ERROR: {POLICY_NAME} is not attached to your role: {role_name}")
    print()
    print("Ask your workshop admin to run:  ./setup/grant_sagemaker_access.sh")
    print("Then re-run this cell.")
    sys.exit(1)

## 2. Review the Deployment Package

The `agentcore_deploy/` directory contains the pre-built agent code:
- `agent.py` — the agent code wrapped in a `BedrockAgentCoreApp` handler
- `pyproject.toml` — Python dependencies

In [ ]:
!ls -la agentcore_deploy/

In [ ]:
!cat agentcore_deploy/agent.py

## 3. Configure and Deploy

AgentCore needs a `.bedrock_agentcore.yaml` config file that specifies the entrypoint, deployment type, and AWS settings. We generate this programmatically to avoid interactive CLI prompts.

The deployment uses CodeBuild to build an ARM64 container — no Docker is required locally.

In [ ]:
import os

import boto3
import yaml
from dotenv import load_dotenv

load_dotenv("../CONFIG.txt")

REGION = os.getenv("REGION", "us-east-1")

sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

agent_dir = os.path.abspath("agentcore_deploy")
entrypoint = os.path.join(agent_dir, "agent.py")

AGENT_NAME = "basic_strands_agent"

config = {
    "default_agent": AGENT_NAME,
    "agents": {
        AGENT_NAME: {
            "name": AGENT_NAME,
            "language": "python",
            "entrypoint": entrypoint,
            "deployment_type": "direct_code_deploy",
            "runtime_type": "PYTHON_3_13",
            "platform": "linux/arm64",
            "source_path": agent_dir,
            "aws": {
                "account": account_id,
                "region": REGION,
                "execution_role_auto_create": True,
                "ecr_auto_create": False,
                "s3_auto_create": True,
                "network_configuration": {
                    "network_mode": "PUBLIC",
                },
                "protocol_configuration": {
                    "server_protocol": "HTTP",
                },
                "observability": {
                    "enabled": True,
                },
            },
        }
    },
}

config_path = os.path.join(agent_dir, ".bedrock_agentcore.yaml")
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Wrote {config_path}")
print(f"  Agent:      {AGENT_NAME}")
print(f"  Account:    {account_id}")
print(f"  Region:     {REGION}")
print(f"  Deploy:     direct_code_deploy")

In [ ]:
# Install zip if not available (needed by direct_code_deploy)
!which zip || (sudo apt-get update -qq && sudo apt-get install -y -qq zip)

# Deploy (--auto-update-on-conflict updates the agent if it already exists)
!cd agentcore_deploy && agentcore deploy --auto-update-on-conflict

## 4. Invoke the Deployed Agent

Once deployed, invoke the agent via the `agentcore` CLI or programmatically with boto3.

In [ ]:
# Invoke via CLI
!cd agentcore_deploy && agentcore invoke '{"prompt": "What time is it and what is 42 + 17?"}'

### Invoke via boto3

For programmatic access, use the `bedrock-agentcore` boto3 client:

In [ ]:
import json
import uuid

import yaml
from botocore.config import Config

with open("agentcore_deploy/.bedrock_agentcore.yaml") as f:
    deploy_config = yaml.safe_load(f)

default_agent = deploy_config["default_agent"]
agent_config = deploy_config["agents"][default_agent]
AGENT_ARN = agent_config["bedrock_agentcore"]["agent_arn"]
AGENT_REGION = agent_config["aws"]["region"]

print(f"Agent:  {default_agent}")
print(f"ARN:    {AGENT_ARN}")
print(f"Region: {AGENT_REGION}")

In [ ]:
agentcore_config = Config(read_timeout=300)


def invoke_agent(prompt: str):
    """Invoke the deployed agent via boto3."""
    client = boto3.client(
        "bedrock-agentcore",
        region_name=AGENT_REGION,
        config=agentcore_config,
    )

    response = client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
        qualifier="DEFAULT",
    )

    content = "".join(
        chunk.decode("utf-8") for chunk in response.get("response", [])
    )

    try:
        return json.loads(content)
    except (json.JSONDecodeError, ValueError):
        return content


result = invoke_agent("What is 100 + 200?")
print(result)

## 5. Cleanup

When you're done, remove the agent from AgentCore Runtime. This deletes the deployed runtime but keeps your local files.

In [ ]:
# Uncomment the line below to destroy the deployed agent
# !cd agentcore_deploy && agentcore destroy